# MOONPIERCER — Full Chord PBH Lunar Crater Search Analysis

This notebook is the single comprehensive analysis workspace for the MOONPIERCER pipeline.
It loads HPC results, generates all figures (saved to `plots/pdf/` and `plots/png/`),
and performs statistical analysis.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure moonpiercer is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from moonpiercer.config import ChordConfig
from moonpiercer.constants import LUNAR_RADIUS_M, LUNAR_SURFACE_AREA_KM2
from moonpiercer.geometry import (
    chord_length_from_separation,
    expected_ellipticity_from_separation,
)
from moonpiercer.velocity import (
    max_physical_angular_offset_deg,
    maxwell_boltzmann_speed_pdf,
    rotation_offset_deg,
)
from moonpiercer.null_model import benjamini_hochberg
from moonpiercer.plotting import (
    plot_annotated_chip,
    plot_chord_map,
    plot_score_distribution,
    plot_spatial_coverage,
    plot_transit_cone_diagram,
)
from moonpiercer.io_utils import (
    PLOTS_PDF_DIR,
    PLOTS_PNG_DIR,
    RESULTS_DIR,
    load_dataframe,
    load_json,
    save_figure,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

print(f"Project root: {PROJECT_ROOT}")
print(f"Plots PDF: {PLOTS_PDF_DIR}")
print(f"Plots PNG: {PLOTS_PNG_DIR}")

## 1. Load HPC Results

In [ ]:
# Update these paths to match your HPC run output
RUN_DIR = RESULTS_DIR / "moonpiercer_run" / "global"

# Load results if available
try:
    summary = load_json(RUN_DIR / "global_summary.json")
    all_pairs = load_dataframe(RUN_DIR / "all_pairs_scored.csv")
    sig_pairs = load_dataframe(RUN_DIR / "significant_pairs.csv")
    null_scores = np.load(RUN_DIR / "null_best_scores.npy")
    print(f"Loaded results from {RUN_DIR}")
    print(f"  Chips: {summary.get('n_chips_with_data', '?')}")
    print(f"  Craters: {summary.get('n_craters', '?')}")
    print(f"  Raw pairs: {summary.get('n_raw_pairs', '?')}")
    print(f"  Significant: {summary.get('n_significant', '?')}")
    HAS_RESULTS = True
except Exception as e:
    print(f"No HPC results found ({e}). Running in demo mode.")
    HAS_RESULTS = False
    all_pairs = pd.DataFrame()
    sig_pairs = pd.DataFrame()
    null_scores = np.array([])
    summary = {}

## 2. Methodology Figures

These figures illustrate the pipeline methodology and can be generated
without HPC results.

In [ ]:
# Figure: Transit Cone Diagram
fig_cone = plot_transit_cone_diagram(
    entry_lon=30.0, entry_lat=15.0,
    ellipticity=1.8, orientation_deg=45.0,
    cone_half_deg=2.0,
)
save_figure(fig_cone, "transit_cone_diagram")
plt.show()

In [ ]:
# Figure: PBH Velocity Distribution
fig_vel, ax = plt.subplots(figsize=(8, 4))
v = np.linspace(0.1, 600, 2000)
pdf = maxwell_boltzmann_speed_pdf(v)
ax.plot(v, pdf, 'k-', lw=1.5)
ax.fill_between(v, pdf, alpha=0.2)
ax.axvline(220, color='red', ls='--', label=r'$v_0 = 220$ km/s')
ax.axvline(544, color='blue', ls='--', label=r'$v_{esc} = 544$ km/s')
ax.set_xlabel('PBH speed [km/s]')
ax.set_ylabel('Probability density')
ax.set_title('PBH Speed Distribution (Standard Halo Model)')
ax.legend()
fig_vel.tight_layout()
save_figure(fig_vel, "velocity_distribution")
plt.show()

In [ ]:
# Figure: Rotation offset vs chord length
fig_rot, ax = plt.subplots(figsize=(8, 4))
seps = np.linspace(30, 180, 200)
L_km = chord_length_from_separation(seps) / 1e3
for v_km_s, color, label in [(50, 'red', '50 km/s'), (100, 'orange', '100 km/s'),
                              (220, 'green', '220 km/s'), (544, 'blue', '544 km/s')]:
    offset = rotation_offset_deg(chord_length_from_separation(seps), v_km_s)
    ax.plot(L_km, offset * 3600, color=color, label=label)  # convert to arcsec

ax.set_xlabel('Chord length [km]')
ax.set_ylabel('Rotation offset [arcsec]')
ax.set_title('Exit Crater Offset from Lunar Rotation')
ax.legend()
ax.set_xlim(0, 3500)
fig_rot.tight_layout()
save_figure(fig_rot, "rotation_offset_vs_chord")
plt.show()

In [ ]:
# Figure: Expected ellipticity vs angular separation
fig_ellip, ax = plt.subplots(figsize=(8, 4))
seps = np.linspace(20, 180, 200)
e = expected_ellipticity_from_separation(seps)
ax.plot(seps, e, 'k-', lw=1.5)
ax.axhline(1.0, color='gray', ls=':', alpha=0.5)
ax.set_xlabel('Angular separation [deg]')
ax.set_ylabel('Expected crater ellipticity')
ax.set_title('Crater Ellipticity vs Chord Angular Separation')
ax.set_ylim(0.9, 6)
fig_ellip.tight_layout()
save_figure(fig_ellip, "ellipticity_vs_separation")
plt.show()

## 3. Detection Statistics

In [ ]:
if HAS_RESULTS:
    # Load global crater catalogue
    CHIP_DIR = RESULTS_DIR / "moonpiercer_run" / "chips"
    crater_files = sorted(CHIP_DIR.glob("chip_*/craters.csv"))
    print(f"Found {len(crater_files)} chip crater files")
    
    craters = pd.concat([load_dataframe(f) for f in crater_files], ignore_index=True)
    print(f"Total craters: {len(craters):,}")
    print(f"\nRadius statistics [m]:")
    print(craters['radius_m'].describe())
    print(f"\nFreshness Index statistics:")
    if 'freshness_index' in craters.columns:
        print(craters['freshness_index'].describe())
else:
    print("No HPC results. Skipping detection statistics.")

In [ ]:
if HAS_RESULTS and 'freshness_index' in craters.columns:
    fig_cov = plot_spatial_coverage(craters)
    save_figure(fig_cov, "spatial_coverage")
    plt.show()

## 4. Pair Analysis

In [ ]:
if HAS_RESULTS and len(all_pairs) > 0:
    print(f"Total scored pairs: {len(all_pairs):,}")
    print(f"Significant pairs (BH-FDR): {len(sig_pairs)}")
    print(f"\nBest pair:")
    best = all_pairs.iloc[0]
    print(f"  Score: {best['score']:.6f}")
    print(f"  Separation: {best['separation_deg']:.4f} deg")
    if 'p_value' in all_pairs.columns:
        print(f"  p-value: {best['p_value']:.6f}")
    
    # Score distribution
    fig_scores = plot_score_distribution(
        all_pairs["score"].to_numpy(), null_scores
    )
    save_figure(fig_scores, "score_distribution")
    plt.show()
    
    # Chord map
    fig_map = plot_chord_map(all_pairs, n_best=20)
    save_figure(fig_map, "chord_map_top20")
    plt.show()
else:
    print("No pair results to analyse.")

## 5. Sensitivity Analysis

In [ ]:
if HAS_RESULTS and len(all_pairs) > 0:
    print("Score component breakdown for top 10 pairs:")
    score_cols = [c for c in all_pairs.columns if c.startswith('T_')]
    display_cols = ['score', 'separation_deg', 'radius_a_m', 'radius_b_m'] + score_cols
    print(all_pairs[display_cols].head(10).to_string(index=False))
else:
    print("No results for sensitivity analysis.")

## 6. Crater Survival Timescale

Compute the survival timescale $\tau_{\rm surv}$ using topographic diffusion.

In [ ]:
from moonpiercer.constants import (
    KAPPA_CLASSICAL_LOW_M2_PER_MYR,
    KAPPA_CLASSICAL_HIGH_M2_PER_MYR,
    KAPPA_ADOPTED_M2_PER_MYR,
)

# Classical diffusion: tau = r^2 / (4 * kappa)
radii_m = np.array([1, 2, 3, 5, 10])  # crater radii in metres
kappa = KAPPA_ADOPTED_M2_PER_MYR

fig_tau, ax = plt.subplots(figsize=(8, 4))
for kap, label, ls in [
    (KAPPA_CLASSICAL_LOW_M2_PER_MYR, r'$\kappa$ = 0.5 m$^2$/Myr (low)', '--'),
    (KAPPA_ADOPTED_M2_PER_MYR, r'$\kappa$ = 1.6 m$^2$/Myr (adopted)', '-'),
    (KAPPA_CLASSICAL_HIGH_M2_PER_MYR, r'$\kappa$ = 5.5 m$^2$/Myr (high)', ':'),
]:
    r_range = np.linspace(0.5, 15, 200)
    tau = r_range**2 / (4 * kap)
    ax.plot(r_range, tau, ls=ls, lw=1.5, label=label)

ax.set_xlabel('Crater radius [m]')
ax.set_ylabel(r'$\tau_{\rm surv}$ [Myr]')
ax.set_title('Crater Survival Timescale (Classical Diffusion)')
ax.legend(fontsize=8)
ax.set_ylim(0, 15)
fig_tau.tight_layout()
save_figure(fig_tau, "crater_survival_timescale")
plt.show()

print(f"\nSurvival times at kappa = {kappa} m^2/Myr:")
for r in radii_m:
    tau = r**2 / (4 * kappa)
    print(f"  r = {r} m: tau = {tau:.2f} Myr")

## 7. Key Physical Parameters Summary

In [ ]:
max_offset = max_physical_angular_offset_deg(v_min_km_s=50.0)
max_offset_m = max_offset * np.pi / 180.0 * LUNAR_RADIUS_M

print("Physical constraints summary:")
print(f"  Max rotation offset (50 km/s, diametral): {max_offset:.5f} deg ({max_offset * 3600:.1f} arcsec, {max_offset_m:.0f} m)")
print(f"  Max rotation offset (220 km/s, diametral): {float(rotation_offset_deg(2*LUNAR_RADIUS_M, 220)):.5f} deg")
print(f"  Adopted hard cut: {ChordConfig().max_chord_deviation_deg} deg")
print(f"  Safety margin: {ChordConfig().max_chord_deviation_deg / max_offset:.1f}x physical max")
print(f"  Lunar surface area: {LUNAR_SURFACE_AREA_KM2:.0f} km^2")
print(f"  Lunar radius: {LUNAR_RADIUS_M:.0f} m")
print(f"  Lunar diameter: {2*LUNAR_RADIUS_M/1e3:.1f} km")